In [130]:
import pandas as pd
from ydata_profiling import ProfileReport

## Importando os dados

In [148]:
features = pd.read_csv("../data/raw/raw_features_com_gc.csv", index_col=0)
metadata_cols = pd.read_csv("../data/processed/metadata_filtrada.csv").set_index("sample")[["source_type", "WHO_Priority"]]

In [132]:
features.head()

,n_genes_cromossomal,n_genes_plasmidial,mdr_score_cromossomal,mdr_score_plasmidial,n_plasmidial_aminoglycoside,n_plasmidial_antimycobacterial,n_plasmidial_beta-lactam,n_plasmidial_biocide_antiseptic,n_plasmidial_fluoroquinolone,n_plasmidial_glycopeptide,...,vfdb_plasmid_regulation,vfdb_plasmid_stress_survival,virulence_score_plasmidial,virulence_score_cromossomal,n_proviruses,tem_provirus_em_plasmidio,tem_amr_em_provirus,gc_cromossomal,gc_plasmidial,gc_diff_plasmid_cromossomo
sample,,,,,,,,,,,,,,,,,,,,,
GCA_000216055.2,2,0,1,0,0,0,0,0,0,0,...,0,0,0,0,4,0,0,34.95,35.13,0.18
GCA_000316425.1,47,1,13,6,0,0,1,1,1,0,...,0,0,4,8,7,0,0,50.57,47.65,-2.92
GCA_000223095.2,6,0,7,0,0,0,0,0,0,0,...,0,0,1,6,3,0,0,47.69,44.64,-3.05
GCA_036761135.1,10,6,10,5,0,0,1,1,0,0,...,0,0,0,4,1,0,0,37.79,38.10,0.31
GCA_003670255.1,7,7,8,5,1,0,3,1,0,0,...,0,0,0,4,4,0,0,37.95,38.85,0.90


In [133]:
features.columns

Index(['n_genes_cromossomal', 'n_genes_plasmidial', 'mdr_score_cromossomal',
       'mdr_score_plasmidial', 'n_plasmidial_aminoglycoside',
       'n_plasmidial_antimycobacterial', 'n_plasmidial_beta-lactam',
       'n_plasmidial_biocide_antiseptic', 'n_plasmidial_fluoroquinolone',
       'n_plasmidial_glycopeptide', 'n_plasmidial_mls',
       'n_plasmidial_nitroimidazole', 'n_plasmidial_other',
       'n_plasmidial_oxazolidinone', 'n_plasmidial_peptide',
       'n_plasmidial_phenicol', 'n_plasmidial_phosphonic_acid',
       'n_plasmidial_pleuromutilin', 'n_plasmidial_rifamycin',
       'n_plasmidial_sulfonamide_diaminopyrimidine',
       'n_plasmidial_tetracycline', 'n_plasmidial_moenomycin_antibiotic',
       'n_plasmidial_mupirocin-like_antibiotic',
       'n_plasmidial_thiosemicarbazone_antibiotic', 'pct_contigs_cromossomal',
       'pct_contigs_plasmidial', 'pct_plasmid_topology_dtr',
       'pct_plasmid_topology_itr', 'pct_plasmid_topology_no_terminal_repeats',
       'pct_plasmid

## Imbalance
Removendo as categorias de genes plasmidiais com >80

In [134]:
tirar = [
        # genes AMR plasmidiais
       'n_plasmidial_antimycobacterial',   
       'n_plasmidial_biocide_antiseptic', 
       'n_plasmidial_nitroimidazole', 
       'n_plasmidial_oxazolidinone', 
       'n_plasmidial_pleuromutilin',
       'n_plasmidial_moenomycin_antibiotic',
       'n_plasmidial_mupirocin-like_antibiotic',
       'n_plasmidial_thiosemicarbazone_antibiotic', 

        # genes de virulência plasmidiais
       'vfdb_plasmid_others', 
       'vfdb_plasmid_stress_survival',

        #skewed
        'vfdb_plasmid_motility'
       ]

In [135]:
novas_features = features.drop(tirar, axis=1)

novas_features.head()

,n_genes_cromossomal,n_genes_plasmidial,mdr_score_cromossomal,mdr_score_plasmidial,n_plasmidial_aminoglycoside,n_plasmidial_beta-lactam,n_plasmidial_fluoroquinolone,n_plasmidial_glycopeptide,n_plasmidial_mls,n_plasmidial_other,...,vfdb_plasmid_nutritional_metabolic_factor,vfdb_plasmid_regulation,virulence_score_plasmidial,virulence_score_cromossomal,n_proviruses,tem_provirus_em_plasmidio,tem_amr_em_provirus,gc_cromossomal,gc_plasmidial,gc_diff_plasmid_cromossomo
sample,,,,,,,,,,,,,,,,,,,,,
GCA_000216055.2,2,0,1,0,0,0,0,0,0,0,...,0,0,0,0,4,0,0,34.95,35.13,0.18
GCA_000316425.1,47,1,13,6,0,1,1,0,0,0,...,0,0,4,8,7,0,0,50.57,47.65,-2.92
GCA_000223095.2,6,0,7,0,0,0,0,0,0,0,...,0,0,1,6,3,0,0,47.69,44.64,-3.05
GCA_036761135.1,10,6,10,5,0,1,0,0,2,0,...,0,0,0,4,1,0,0,37.79,38.10,0.31
GCA_003670255.1,7,7,8,5,1,3,0,0,0,0,...,0,0,0,4,4,0,0,37.95,38.85,0.90


## Missing
Como alguns genomas não possuem plasmidio, o %GC fica Nan.
- Manter apenas a coluna 'gc_diff_plasmid_cromossomo'

In [136]:
novas_features = novas_features.drop(['gc_cromossomal', 'gc_plasmidial'], axis=1)

novas_features.head()

,n_genes_cromossomal,n_genes_plasmidial,mdr_score_cromossomal,mdr_score_plasmidial,n_plasmidial_aminoglycoside,n_plasmidial_beta-lactam,n_plasmidial_fluoroquinolone,n_plasmidial_glycopeptide,n_plasmidial_mls,n_plasmidial_other,...,vfdb_plasmid_immune_modulation,vfdb_plasmid_invasion,vfdb_plasmid_nutritional_metabolic_factor,vfdb_plasmid_regulation,virulence_score_plasmidial,virulence_score_cromossomal,n_proviruses,tem_provirus_em_plasmidio,tem_amr_em_provirus,gc_diff_plasmid_cromossomo
sample,,,,,,,,,,,,,,,,,,,,,
GCA_000216055.2,2,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,4,0,0,0.18
GCA_000316425.1,47,1,13,6,0,1,1,0,0,0,...,0,0,0,0,4,8,7,0,0,-2.92
GCA_000223095.2,6,0,7,0,0,0,0,0,0,0,...,0,0,0,0,1,6,3,0,0,-3.05
GCA_036761135.1,10,6,10,5,0,1,0,0,2,0,...,0,0,0,0,0,4,1,0,0,0.31
GCA_003670255.1,7,7,8,5,1,3,0,0,0,0,...,0,0,0,0,0,4,4,0,0,0.90


In [137]:
novas_features['gc_diff_plasmid_cromossomo'].isna().sum()

np.int64(809)

In [138]:
novas_features['gc_diff_plasmid_cromossomo'] = novas_features['gc_diff_plasmid_cromossomo'].fillna(0)

novas_features['gc_diff_plasmid_cromossomo'].isna().sum()

np.int64(0)

## Correlation

In [139]:
novas_features.head()

,n_genes_cromossomal,n_genes_plasmidial,mdr_score_cromossomal,mdr_score_plasmidial,n_plasmidial_aminoglycoside,n_plasmidial_beta-lactam,n_plasmidial_fluoroquinolone,n_plasmidial_glycopeptide,n_plasmidial_mls,n_plasmidial_other,...,vfdb_plasmid_immune_modulation,vfdb_plasmid_invasion,vfdb_plasmid_nutritional_metabolic_factor,vfdb_plasmid_regulation,virulence_score_plasmidial,virulence_score_cromossomal,n_proviruses,tem_provirus_em_plasmidio,tem_amr_em_provirus,gc_diff_plasmid_cromossomo
sample,,,,,,,,,,,,,,,,,,,,,
GCA_000216055.2,2,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,4,0,0,0.18
GCA_000316425.1,47,1,13,6,0,1,1,0,0,0,...,0,0,0,0,4,8,7,0,0,-2.92
GCA_000223095.2,6,0,7,0,0,0,0,0,0,0,...,0,0,0,0,1,6,3,0,0,-3.05
GCA_036761135.1,10,6,10,5,0,1,0,0,2,0,...,0,0,0,0,0,4,1,0,0,0.31
GCA_003670255.1,7,7,8,5,1,3,0,0,0,0,...,0,0,0,0,0,4,4,0,0,0.90


### Dividindo o n. de genes de cada classe pelo total para remover a coluna do total

In [140]:
plasmidial_cols = [c for c in novas_features.columns if c.startswith("n_plasmidial_")]

plasmidial_cols

['n_plasmidial_aminoglycoside',
 'n_plasmidial_beta-lactam',
 'n_plasmidial_fluoroquinolone',
 'n_plasmidial_glycopeptide',
 'n_plasmidial_mls',
 'n_plasmidial_other',
 'n_plasmidial_peptide',
 'n_plasmidial_phenicol',
 'n_plasmidial_phosphonic_acid',
 'n_plasmidial_rifamycin',
 'n_plasmidial_sulfonamide_diaminopyrimidine',
 'n_plasmidial_tetracycline']

In [141]:
for coluna in plasmidial_cols:
    novas_features[f'prop_{coluna}'] = novas_features[coluna] / novas_features["n_genes_plasmidial"]

novas_features.head()

,n_genes_cromossomal,n_genes_plasmidial,mdr_score_cromossomal,mdr_score_plasmidial,n_plasmidial_aminoglycoside,n_plasmidial_beta-lactam,n_plasmidial_fluoroquinolone,n_plasmidial_glycopeptide,n_plasmidial_mls,n_plasmidial_other,...,prop_n_plasmidial_fluoroquinolone,prop_n_plasmidial_glycopeptide,prop_n_plasmidial_mls,prop_n_plasmidial_other,prop_n_plasmidial_peptide,prop_n_plasmidial_phenicol,prop_n_plasmidial_phosphonic_acid,prop_n_plasmidial_rifamycin,prop_n_plasmidial_sulfonamide_diaminopyrimidine,prop_n_plasmidial_tetracycline
sample,,,,,,,,,,,,,,,,,,,,,
GCA_000216055.2,2,0,1,0,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GCA_000316425.1,47,1,13,6,0,1,1,0,0,0,...,1.0,0.0,0.000000,0.0,0.0,1.000000,0.0,1.0,0.000000,1.000000
GCA_000223095.2,6,0,7,0,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GCA_036761135.1,10,6,10,5,0,1,0,0,2,0,...,0.0,0.0,0.333333,0.0,0.0,0.000000,0.0,0.0,0.166667,0.166667
GCA_003670255.1,7,7,8,5,1,3,0,0,0,0,...,0.0,0.0,0.000000,0.0,0.0,0.142857,0.0,0.0,0.142857,0.000000


In [142]:
prop_cols = [f"prop_{c}" for c in plasmidial_cols]
novas_features[prop_cols] = novas_features[prop_cols].fillna(0)

novas_features.head()

,n_genes_cromossomal,n_genes_plasmidial,mdr_score_cromossomal,mdr_score_plasmidial,n_plasmidial_aminoglycoside,n_plasmidial_beta-lactam,n_plasmidial_fluoroquinolone,n_plasmidial_glycopeptide,n_plasmidial_mls,n_plasmidial_other,...,prop_n_plasmidial_fluoroquinolone,prop_n_plasmidial_glycopeptide,prop_n_plasmidial_mls,prop_n_plasmidial_other,prop_n_plasmidial_peptide,prop_n_plasmidial_phenicol,prop_n_plasmidial_phosphonic_acid,prop_n_plasmidial_rifamycin,prop_n_plasmidial_sulfonamide_diaminopyrimidine,prop_n_plasmidial_tetracycline
sample,,,,,,,,,,,,,,,,,,,,,
GCA_000216055.2,2,0,1,0,0,0,0,0,0,0,...,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000
GCA_000316425.1,47,1,13,6,0,1,1,0,0,0,...,1.0,0.0,0.000000,0.0,0.0,1.000000,0.0,1.0,0.000000,1.000000
GCA_000223095.2,6,0,7,0,0,0,0,0,0,0,...,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000
GCA_036761135.1,10,6,10,5,0,1,0,0,2,0,...,0.0,0.0,0.333333,0.0,0.0,0.000000,0.0,0.0,0.166667,0.166667
GCA_003670255.1,7,7,8,5,1,3,0,0,0,0,...,0.0,0.0,0.000000,0.0,0.0,0.142857,0.0,0.0,0.142857,0.000000


In [143]:
novas_features = novas_features.drop(plasmidial_cols, axis=1)

novas_features = novas_features.drop('n_genes_plasmidial', axis=1)

### Tirando colunas redundantes

In [147]:
tirar_redundante = ['pct_contigs_cromossomal', 'pct_plasmid_topology_no_terminal_repeats']

final_features = novas_features.drop(tirar_redundante, axis=1)

final_features.head()

,n_genes_cromossomal,mdr_score_cromossomal,mdr_score_plasmidial,pct_contigs_plasmidial,pct_plasmid_topology_dtr,pct_plasmid_topology_itr,pct_plasmidial_amr,pct_plasmidial_conjugacao_amr,n_genes_plasmidial_conjugativo,vfdb_plasmid_adherence,...,prop_n_plasmidial_fluoroquinolone,prop_n_plasmidial_glycopeptide,prop_n_plasmidial_mls,prop_n_plasmidial_other,prop_n_plasmidial_peptide,prop_n_plasmidial_phenicol,prop_n_plasmidial_phosphonic_acid,prop_n_plasmidial_rifamycin,prop_n_plasmidial_sulfonamide_diaminopyrimidine,prop_n_plasmidial_tetracycline
sample,,,,,,,,,,,,,,,,,,,,,
GCA_000216055.2,2,1,0,1.655629,0.000000,0.0,0.000000,0.000000,0,0,...,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000
GCA_000316425.1,47,13,6,5.675676,4.761905,0.0,4.761905,0.000000,0,6,...,1.0,0.0,0.000000,0.0,0.0,1.000000,0.0,1.0,0.000000,1.000000
GCA_000223095.2,6,7,0,6.349206,0.000000,0.0,0.000000,0.000000,0,0,...,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000
GCA_036761135.1,10,10,5,18.269231,0.000000,0.0,10.526316,2.631579,1,0,...,0.0,0.0,0.333333,0.0,0.0,0.000000,0.0,0.0,0.166667,0.166667
GCA_003670255.1,7,8,5,22.406639,1.851852,0.0,9.259259,1.851852,1,0,...,0.0,0.0,0.000000,0.0,0.0,0.142857,0.0,0.0,0.142857,0.000000


## Verificando com ydata

In [149]:
df_ydata = final_features.join(metadata_cols)
profile = ProfileReport(df_ydata, title="Features (pós-estruturação-1)", minimal=False)
profile.to_file("../reports/features_depois1_ydata.html")

Export report to file: 100%|██████████| 1/1 [00:00<00:00,  2.68it/s]
